# 05. Keep correct teacher answers and build the SFT files

Run after `04_label_with_teacher.py`. Parses the teacher verdict from each completion, keeps only the instances where it matches the gold label and writes the prompt/completion files used for fine-tuning.

Inputs: `../data/training/{train,validation}_gold_plus_web_search_completions.jsonl`.

Outputs: `../data/training/{train,validation}_gold_plus_web_search_correct_only.jsonl` and the matching `*_correct_only_sft.jsonl` (fine-tuning format).


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "../data/training/"

In [ ]:
df_train_labeled = pd.read_json(
    os.path.join(OUTPUT_DIR, "train_gold_plus_web_search_completions.jsonl"), lines=True)
print(df_train_labeled.shape)
df_train_labeled.head()

In [ ]:
df_train_labeled.completion[0]

In [ ]:
df_validation_labeled = pd.read_json(
    os.path.join(OUTPUT_DIR, "validation_gold_plus_web_search_completions.jsonl"), lines=True)
print(df_validation_labeled.shape)
df_validation_labeled.head()

In [ ]:
import re

def map_verdict_to_label(label):
    """Map a verdict to the label names used in the dataset: supports, nei or refutes."""
    label = label.upper().strip()
    if label == "SUPPORTS" or label == "SUPPORTED" or label== "TRUE":
        return "supports"
    elif label == "NEI":
        return "nei"
    elif label == "REFUTES" or label == "REFUTED" or label == "FALSE":
        return "refutes"


def map_true_false_to_label(response):
    """Map True/False responses to SUPPORTS/REFUTES labels."""
    response = response.upper().strip()
    if response == "TRUE":
        return "SUPPORTS"
    elif response == "FALSE":
        return "REFUTES"
    return response  # Return original if not TRUE or FALSE


def extract_verification(full_response):
    """Extract the teacher verdict from a completion and return it as a dataset label.
    TRUE/FALSE answers are mapped to supports/refutes. Returns None when no verdict is found.
    """
    # Extract the answer between <answer> and </answer> tags after </thought>
    match = re.search(
        r"</thought>.*?<answer>(SUPPORTED|SUPPORTS|NEI|REFUTES|REFUTED|TRUE|FALSE)</answer>",
        full_response,
        re.DOTALL | re.IGNORECASE,
    )
    if match:
        label = map_true_false_to_label(match.group(1))
        return map_verdict_to_label(label)

    # Fallback: find any <answer> tag with the labels
    match = re.search(
        r"<answer>(SUPPORTED|SUPPORTS|NEI|REFUTES|REFUTED|TRUE|FALSE)</answer>",
        full_response,
        re.IGNORECASE,
    )
    if match:
        label = map_true_false_to_label(match.group(1))
        return map_verdict_to_label(label)

    # Numeric answers are returned as is
    match = re.search(r"<answer>([012])</answer>", full_response)
    if match:
        return match.group(1)

    lines = full_response.strip().split("\n")

    # Look for any SUPPORTS, NEI, or REFUTES in the last few lines
    for line in reversed(lines[-20:] if len(lines) > 20 else lines):
        line = line.strip().upper()
        if "SUPPORTS" in line or "TRUE" in line or "SUPPORTED" in line:
            return map_verdict_to_label("SUPPORTS")
        elif "NEI" in line:
            # Skip NEI lines and keep looking for other labels
            continue
        elif "REFUTES" in line or "FALSE" in line or "REFUTED" in line:
            return map_verdict_to_label("REFUTES")

    # No verdict found: the instance will not match its gold label and is dropped
    print("Could not extract verification result from response")
    return None

In [ ]:
df_train_labeled["gemini_label"] = df_train_labeled.completion.apply(lambda x: extract_verification(x))
df_validation_labeled["gemini_label"] = df_validation_labeled.completion.apply(lambda x: extract_verification(x))

In [ ]:
df_train_labeled.tail(5)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(df_train_labeled["label"], df_train_labeled["gemini_label"], zero_division=0, digits=4))
print(classification_report(df_validation_labeled["label"], df_validation_labeled["gemini_label"], zero_division=0, digits=4))

In [ ]:
# Inspect instances where the teacher verdict differs from the gold label
def show_misclassified_examples(df, label_col="gemini_label", true_label_col="label", n=5):
    """Show misclassified examples from the DataFrame."""
    misclassified = df[df[label_col] != df[true_label_col]]
    if misclassified.empty:
        print("No misclassified examples found.")
        return
    
    print(f"Found {len(misclassified)} misclassified examples. Showing {min(n, len(misclassified))} samples:")
    for i, row in misclassified.sample(n=min(n, len(misclassified))).iterrows():
        print(f"\nExample {i+1}:")
        print(f"Claim: {row['claim']}")
        print(f"Evidence Type: {row['evidence_type']}")
        print(f"Evidence: {row['evidence']}")
        print(f"Predicted: {row[label_col]}")
        print(f"True Label: {row[true_label_col]}")
        print(f"Response: {row['completion']}")
        
show_misclassified_examples(df_train_labeled, label_col="gemini_label", true_label_col="label", n=5)

In [ ]:
def save_correct_only(df, original_path):
    """
    Filter rows where gemini_label matches label and save to a new file
    with '_correct_only' added to the filename.
    """
    # Filter for correct predictions
    df_correct = df[df['gemini_label'] == df['label']].copy()
    print(f"Filtered: {len(df_correct)} / {len(df)} rows are correct.")

    # Generate new filename
    base, ext = os.path.splitext(original_path)
    new_path = f"{base.replace('_completions', '')}_correct_only{ext}"

    # Save to file
    df_correct.to_json(new_path, orient="records", lines=True)
    print(f"Saved correct-only dataset to: {new_path}")
    return df_correct, new_path

df_train_correct, train_path = save_correct_only(df_train_labeled, os.path.join(OUTPUT_DIR, "train_gold_plus_web_search_completions.jsonl"))
df_val_correct, val_path = save_correct_only(df_validation_labeled, os.path.join(OUTPUT_DIR, "validation_gold_plus_web_search_completions.jsonl"))

# Save only the prompt and completion columns, with _sft in the filename
def save_sft_format(df, original_path):
    """
    Save DataFrame in SFT format with only 'prompt' and 'completion' columns.
    Adds '_sft' to the filename.
    """
    # Create new DataFrame with only 'prompt' and 'completion'
    df_sft = df[['prompt', 'completion',]].copy()

    # Generate new filename
    base, ext = os.path.splitext(original_path)
    new_path = f"{base}_sft{ext}"

    # Save to file
    df_sft.to_json(new_path, orient="records", lines=True)
    print(f"Saved SFT format dataset to: {new_path}")
    return df_sft, new_path
df_train_sft, train_sft_path = save_sft_format(df_train_correct, train_path)
df_val_sft, val_sft_path = save_sft_format(df_val_correct, val_path)

In [ ]:
df_train_correct.label.value_counts(dropna=False, normalize=True)